# Solar Filament Segmentation Challenge 2026 — End-to-End Reproducible Pipeline

**Team:** `MXlop` (Timur Shakurov)  
**Leaderboard Result:** Private PQ `0.39` (Top 12) | Public PQ `0.41` | 5-Fold OOF PQ `0.4683` | Strict Nested LOFO PQ `0.4676`  
**Disjointness Guarantee:** `overlapping_pixels = 0` (`many_to_one = 0`) across 100% of OOF and Test images

This notebook demonstrates the complete **zero-retraining** inference pipeline from raw `2048x2048` GONG H-alpha JPEG observations to verified, strictly pixel-disjoint COCO RLE masks in `submission.csv`:
1. **Automatic Public Weight Retrieval** from Hugging Face Hub (`CToraq/solar-filament-segmentation-2026`, 72 files / 190 trained models).
2. **Stage 0 — Astrophysical H-alpha Calibration:** 720-ray IRLS solar limb fit `(cx, cy, R)`, 256-annuli equal-$u=(r/R)^2$ Center-to-Limb Variation (CLV) division, 2-pass feature-masked cloud flat-fielding, and multi-scale DoG ribbon enhancement at exact `1:1` native scale (`1856x1856` centered disk crop retaining 100.000% of all ground-truth pixels).
3. **Stage 1 — Multi-Paradigm Native Ensemble:** 5 calibrated dense dual-head families (`25` checkpoints x 8-view D4 TTA = `200` views/pixel) with exact Law-of-Total-Variance `fstd` inter-family disagreement parity + 5-fold `YOLO11l-seg` (`1536`, 10 consolidated views) instance detector.
4. **Stage 1.5 — Component Tree & Hurdle Antichain DP:** 16-level strictly subset-monotone Component Tree + 77-feature 120-model cross-fitted Hurdle GBDT (`LightGBM + CatBoost + XGBoost`) + hierarchy-consistent Two-Tier Antichain DP (`strict_tau = 0.2600`, `rescue_tau = 0.2275`).
5. **Stage 2 — Oriented Conditional Refinement:** 40-model multi-generation PCA-aligned `384x128` 5-channel refiner ensemble (`160` views/filament) + `join3` MST gap bridging + survivor-only `argmax` disjoint reconstruction & canonical column-major COCO RLE encoding.

In [ ]:
import json
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_util

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'src').exists() and (ROOT_DIR.parent / 'src').exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Configure dataset and weights locations (override via environment variables if running on another machine)
TEST_IMG_DIR = Path(os.environ.get('FILSEG_TEST_DIR', str(ROOT_DIR / 'MAGFiLO_1.0_Kaggle_2026' / 'test' / 'test_images')))
WEIGHTS_DIR = Path(os.environ.get('FILSEG_WEIGHTS_DIR', str(ROOT_DIR / 'work')))
HF_REPO_ID = os.environ.get('FILSEG_HF_REPO', 'CToraq/solar-filament-segmentation-2026')

print(f'ROOT_DIR    : {ROOT_DIR}')
print(f'TEST_IMG_DIR: {TEST_IMG_DIR} (exists={TEST_IMG_DIR.exists()})')
print(f'WEIGHTS_DIR : {WEIGHTS_DIR}')
print(f'PyTorch     : {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

## 1. Download & Verify All Pre-Trained Model Weights (No Authentication Required)

All 72 checkpoint & configuration files (25 Stage-1 dense models, 5 YOLO11l-seg folds, 120 Hurdle GBDTs in `hurdle_v3_models.pkl`, 40 Stage-2 oriented refiners, and `stage2_v3_infer_config.json`) are hosted publicly on Hugging Face Hub (`CToraq/solar-filament-segmentation-2026`). Already-cached local files are skipped automatically.

In [ ]:
from scripts.download_weights import download_weights, verify_weights

download_weights(repo_id=HF_REPO_ID, weights_dir=WEIGHTS_DIR)
total_files, missing_files = verify_weights(WEIGHTS_DIR)
assert not missing_files, f'Missing weights: {missing_files[:5]}'
print(f'Verified {total_files}/{total_files} weight & config artifacts ready for zero-retraining inference.')

## 2. Stage 0 — Astrophysical GONG H-alpha Disk Calibration & Flat-Fielding

Each `2048x2048` raw GONG observation undergoes strictly per-image physical calibration (zero cross-image statistics or external metadata): 720-ray IRLS limb fitting, 256-bin equal-$u=(r/R)^2$ radial CLV normalization, 2-pass feature-masked cloud flat-fielding, and isotropic multi-scale DoG ribbon enhancement inside the native `1856x1856` centered disk crop.

In [ ]:
from src.config import CROP_OFFSET, CROP_SIZE, list_stems_and_paths
from src.solar_prep import compute_physics_channels

test_pairs = list_stems_and_paths(TEST_IMG_DIR)
print(f'Discovered {len(test_pairs)} raw H-alpha test images.')
demo_stem, demo_path = test_pairs[0]
raw_gray = cv2.imread(str(demo_path), cv2.IMREAD_GRAYSCALE)
x_physics, disk_meta = compute_physics_channels(raw_gray)
print(f'Demo image: {demo_stem} | Disk geometry: {disk_meta}')

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
raw_crop = raw_gray[CROP_OFFSET:CROP_OFFSET + CROP_SIZE, CROP_OFFSET:CROP_OFFSET + CROP_SIZE]
panels = [
    (raw_crop, 'Raw GONG H-alpha (1856^2 crop)', 'gray'),
    (x_physics[:, :, 0], 'Ch0: CLV + 2-Pass Flat-Field', 'gray'),
    (x_physics[:, :, 1], 'Ch1: Multi-Scale DoG Absorption', 'inferno'),
    (x_physics[:, :, 2], 'Ch2: Heliocentric mu = cos(theta)', 'viridis'),
]
for ax, (im, title, cmap) in zip(axes, panels):
    ax.imshow(cv2.resize(im, (512, 512), interpolation=cv2.INTER_AREA), cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. End-to-End Zero-Retraining Inference (Stages 1 -> 1.5 -> 2)

- Set `RUN_FULL_TEST_SET = False` (default) to run the complete **190-model** ensemble on a quick 2-image visual demonstration (finishes in ~60 seconds on a single RTX 3090 GPU).
- Set `RUN_FULL_TEST_SET = True` to run inference across all `180` test observations and regenerate the complete `1,196`-row `submission_reproduced.csv` from scratch.

In [ ]:
from src.infer_pipeline import run_end_to_end_inference

RUN_FULL_TEST_SET = False  # Set True to reproduce full 180-image submission.csv end-to-end

target_stems = None if RUN_FULL_TEST_SET else [test_pairs[0][0], test_pairs[min(10, len(test_pairs) - 1)][0]]
out_csv_path = ROOT_DIR / ('submission_reproduced.csv' if RUN_FULL_TEST_SET else 'work/demo_2img_predictions.csv')

infer_out = run_end_to_end_inference(
    test_dir=TEST_IMG_DIR,
    weights_dir=WEIGHTS_DIR,
    out_csv=out_csv_path,
    stems_filter=target_stems,
    return_intermediates=not RUN_FULL_TEST_SET,
)
print(f"Inference complete: {infer_out['total_rows']} masks across {len(infer_out['stems'])} image(s) in {infer_out['elapsed_sec']}s -> {infer_out['out_csv']}")

## 4. Visual Inspection of Intermediate Probability Maps & Final Disjoint Segmentations

Each predicted filament instance is rendered in a distinct color on the flat-fielded H-alpha disk, verifying clean thin-ribbon boundaries and zero pixel overlap (`cover.max() <= 1`).

In [ ]:
if 'intermediates' in infer_out:
    cmap_tab = plt.get_cmap('tab20')
    for stem in infer_out['stems']:
        inter = infer_out['intermediates'][stem]
        rles = infer_out['pred_rles_by_stem'][stem]
        ff = inter['x_u8'][:, :, 0]
        fg = inter['fg_blend']
        dis = inter['disagree']

        overlay = np.repeat(ff[:, :, None], 3, axis=2).astype(np.float32)
        cover = np.zeros((2048, 2048), dtype=np.int32)
        for idx_m, rle in enumerate(rles):
            m = mask_util.decode({'size': [2048, 2048], 'counts': rle['counts'].encode('ascii')})
            cover += m
            mc = m[CROP_OFFSET:CROP_OFFSET + CROP_SIZE, CROP_OFFSET:CROP_OFFSET + CROP_SIZE] > 0
            rgb = np.array(cmap_tab(idx_m % 20)[:3], dtype=np.float32) * 255.0
            overlay[mc] = 0.35 * overlay[mc] + 0.65 * rgb
        assert int(cover.max()) <= 1, 'Overlap detected!'

        fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
        axes[0].imshow(cv2.resize(fg, (512, 512)), cmap='magma', vmin=0, vmax=1)
        axes[0].set_title(f'{stem}: Stage-1 Blended fg ({inter["n_tree_nodes"]} tree nodes)')
        axes[1].imshow(cv2.resize(dis, (512, 512)), cmap='viridis')
        axes[1].set_title(f'LTV fstd-Corrected Inter-Family Disagreement (YOLO={inter["yolo_count"]})')
        axes[2].imshow(cv2.resize(np.clip(overlay, 0, 255).astype(np.uint8), (512, 512)))
        axes[2].set_title(f'Final Stage-2 Disjoint Masks (N={len(rles)}, max_cover={int(cover.max())})')
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

## 5. Strict 14-Check Competition Submission Validation

Verifies exact CSV schema (`filament_id,segmentation_rle`), canonical `[2048, 2048]` column-major COCO RLE round-trip identity, `overlapping_pixels == 0`, and non-empty area bounds on the submitted Kaggle file (`submission/submission.csv` or freshly reproduced full CSV).

In [ ]:
from src.validate_submission import validate_submission_file

sub_to_check = out_csv_path if RUN_FULL_TEST_SET else (ROOT_DIR / 'submission' / 'submission.csv')
val_report = validate_submission_file(sub_to_check, test_dir=TEST_IMG_DIR if TEST_IMG_DIR.exists() else None)
print(json.dumps(val_report, indent=2))
assert val_report['valid'] and val_report['stats']['overlapping_pixels'] == 0

if RUN_FULL_TEST_SET and (ROOT_DIR / 'submission' / 'submission.csv').exists():
    ref_df = pd.read_csv(ROOT_DIR / 'submission' / 'submission.csv')
    new_df = pd.read_csv(out_csv_path)
    print(f'Exact Kaggle ref rows: {len(ref_df)} | Reproduced rows: {len(new_df)}')
    print(f'Exact byte match: {Path(out_csv_path).read_bytes() == (ROOT_DIR / "submission" / "submission.csv").read_bytes()}')

## 6. Full Training & Unit-Test Reproduction Reference

To retrain every model from scratch on `MAGFiLO_1.0_Kaggle_2026` (5-fold observation-grouped CV, zero leakage):
```bash
python src/solar_prep.py
for fam in cnx_s effv2_s upp_r34d manet_eb4 upp_cnx_t; do
    python src/train_stage1_dense.py --family $fam
done
python src/train_stage1_yolo.py
python src/hurdle_scorer.py --save-models work/hurdle_v3_models.pkl
python src/stage2_refiner.py --ckpt-subdir ckpt_stage2_v3 --save-infer-artifacts
```
Unit test suite (metric parity, subset-monotone component tree, LTV `fstd` parity, two-tier Antichain DP, and Stage-2 affine geometry):
```bash
python tests/test_core_infra.py
python tests/test_hierarchy_dp.py
python tests/test_stage2_geometry.py
```